# F4 — Order Book Depth at Spike (W3/W4)

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [2]:
LEVELS = 10
bid_px_cols = [f'bid_px_0{i}' if i < 10 else f'bid_px_{i}' for i in range(LEVELS)]
ask_px_cols = [f'ask_px_0{i}' if i < 10 else f'ask_px_{i}' for i in range(LEVELS)]
bid_sz_cols = [f'bid_sz_0{i}' if i < 10 else f'bid_sz_{i}' for i in range(LEVELS)]
ask_sz_cols = [f'ask_sz_0{i}' if i < 10 else f'ask_sz_{i}' for i in range(LEVELS)]
ALL_COLS = ['ts_event','symbol'] + bid_px_cols + ask_px_cols + bid_sz_cols + ask_sz_cols

# fix column names: they use 2 digits like bid_px_00..09
bid_px_cols = [f'bid_px_{i:02d}' for i in range(LEVELS)]
ask_px_cols = [f'ask_px_{i:02d}' for i in range(LEVELS)]
bid_sz_cols = [f'bid_sz_{i:02d}' for i in range(LEVELS)]
ask_sz_cols = [f'ask_sz_{i:02d}' for i in range(LEVELS)]
ALL_COLS = ['ts_event','symbol'] + bid_px_cols + ask_px_cols + bid_sz_cols + ask_sz_cols


In [4]:
def get_book_snapshot(wm, day, sym, t_snap):
    fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
    if not fpath: return None
    df = pd.read_parquet(fpath[0], columns=ALL_COLS)
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    snap_mask = df['ts_event'].sub(t_snap).abs() < pd.Timedelta('5s')
    snap = df[snap_mask]
    if snap.empty: return None
    snap_row = snap.iloc[snap['ts_event'].sub(t_snap).abs().argmin()]
    del df; gc.collect()
    return snap_row

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
row_configs = [
    (WINDOWS_META['W3'], '2025-03-14', 'W3 Mar-14'),
    (WINDOWS_META['W4'], '2025-06-13', 'W4 Jun-13'),
]
for row_i, (wm, day, label) in enumerate(row_configs):
    t14 = pd.Timestamp(f'{day} 14:00:00', tz='UTC')
    for col_i, (sym, offset_s) in enumerate([(wm['front'], -5),(wm['back'], -5),
                                              (wm['front'], +5),(wm['back'], +5)]):
        ax = axes[row_i][col_i]
        t_snap = t14 + pd.Timedelta(seconds=offset_s)
        snap = get_book_snapshot(wm, day, sym, t_snap)
        if snap is None:
            ax.text(0.5,0.5,'Missing', ha='center', transform=ax.transAxes)
            continue
        bid_prices = [snap[c] for c in bid_px_cols]
        ask_prices = [snap[c] for c in ask_px_cols]
        bid_sizes  = [snap[c] for c in bid_sz_cols]
        ask_sizes  = [snap[c] for c in ask_sz_cols]
        ax.barh(range(LEVELS), bid_sizes, color='#2ecc71', alpha=0.8, label='Bid')
        ax.barh(range(LEVELS), [-s for s in ask_sizes], color='#e74c3c', alpha=0.8, label='Ask')
        ax.axvline(0, color='black', lw=0.5)
        title = f'{label} {sym}\n{"−5s" if offset_s<0 else "+5s"} @ 14:00'
        ax.set_title(title, fontsize=8)
        ax.set_yticks(range(LEVELS))
        ax.set_yticklabels([f'L{i}' for i in range(LEVELS)], fontsize=7)
        if col_i == 0:
            ax.set_ylabel('Book Level')
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=7)
fig.suptitle('Order Book Depth at 14:00 UTC Spike: Before (−5s) vs After (+5s)', fontsize=13)
save_fig(fig, 'F4_orderbook_depth_at_spike.png')


/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_8671/1637489075.py:33: RuntimeWarning: overflow encountered in scalar negative
  ax.barh(range(LEVELS), [-s for s in ask_sizes], color='#e74c3c', alpha=0.8, label='Ask')
/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_8671/1637489075.py:33: RuntimeWarning: overflow encountered in scalar negative
  ax.barh(range(LEVELS), [-s for s in ask_sizes], color='#e74c3c', alpha=0.8, label='Ask')
/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_8671/1637489075.py:33: RuntimeWarning: overflow encountered in scalar negative
  ax.barh(range(LEVELS), [-s for s in ask_sizes], color='#e74c3c', alpha=0.8, label='Ask')
/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_8671/1637489075.py:33: RuntimeWarning: overflow encountered in scalar negative
  ax.barh(range(LEVELS), [-s for s in ask_sizes], color='#e74c3c', alpha=0.8, label='Ask')
/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_8671/1637489075.py:33: Ru

  Saved --> figures/F4_orderbook_depth_at_spike.png
